# Data loading and Cleaning

In [5]:
import numpy as np
import pandas as pd
import seaborn as sns
import glob
import os
import joblib

In [3]:
# loading
assets = joblib.load("nids_iso_forest_pipeline.pkl")
iso_model = assets["iso_model"]
selected_thresh = assets["selected_thresh"]
selected_cols = assets["selected_cols"]
scaler = assets["scaler"]

data = np.load("scaled_datasets.npz")
X_train_scaled = data["X_train"]
X_val_scaled = data["X_val"]
X_test_scaled = data["X_test"]
y_train, y_val, y_test = data["y_train"], data["y_val"], data["y_test"]

print("laoding done")

laoding done


In [6]:
fast_raw_file = "../data/raw/merged_raw.parquet"

if os.path.exists(fast_raw_file):
    print("Loading fast merged raw data...")
    df = pd.read_parquet(fast_raw_file)

else:
    print("Loading CSV files for the first time...")
    path = "../data/raw/*.csv"
    all_files = glob.glob(path)

    df = pd.concat((pd.read_csv(f) for f in all_files), ignore_index=True)

    print("Saving merged data to Parquet in raw folder...")
    df.to_parquet(fast_raw_file, index=False)

print(f"Data successfully loaded! Shape: {df.shape}")

Loading fast merged raw data...
Data successfully loaded! Shape: (2830743, 79)


In [7]:
df.shape

(2830743, 79)

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2830743 entries, 0 to 2830742
Data columns (total 79 columns):
 #   Column                        Dtype  
---  ------                        -----  
 0    Destination Port             int64  
 1    Flow Duration                int64  
 2    Total Fwd Packets            int64  
 3    Total Backward Packets       int64  
 4   Total Length of Fwd Packets   int64  
 5    Total Length of Bwd Packets  int64  
 6    Fwd Packet Length Max        int64  
 7    Fwd Packet Length Min        int64  
 8    Fwd Packet Length Mean       float64
 9    Fwd Packet Length Std        float64
 10  Bwd Packet Length Max         int64  
 11   Bwd Packet Length Min        int64  
 12   Bwd Packet Length Mean       float64
 13   Bwd Packet Length Std        float64
 14  Flow Bytes/s                  float64
 15   Flow Packets/s               float64
 16   Flow IAT Mean                float64
 17   Flow IAT Std                 float64
 18   Flow IAT Max                 int

In [9]:
df.describe()

c:\Users\user\anaconda3\envs\ai_net_env\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
c:\Users\user\anaconda3\envs\ai_net_env\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
count,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,...,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06
mean,8.071483e+03,1.478566e+07,9.361160e+00,1.039377e+01,5.493024e+02,1.616264e+04,2.075999e+02,1.871366e+01,5.820194e+01,6.891013e+01,...,5.418218e+00,-2.741688e+03,8.155132e+04,4.113412e+04,1.531825e+05,5.829582e+04,8.316037e+06,5.038439e+05,8.695752e+06,7.920031e+06
std,1.828363e+04,3.365374e+07,7.496728e+02,9.973883e+02,9.993589e+03,2.263088e+06,7.171848e+02,6.033935e+01,1.860912e+02,2.811871e+02,...,6.364257e+02,1.084989e+06,6.485999e+05,3.933815e+05,1.025825e+06,5.770923e+05,2.363008e+07,4.602984e+06,2.436689e+07,2.336342e+07
min,0.000000e+00,-1.300000e+01,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,-5.368707e+08,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,5.300000e+01,1.550000e+02,2.000000e+00,1.000000e+00,1.200000e+01,0.000000e+00,6.000000e+00,0.000000e+00,6.000000e+00,0.000000e+00,...,0.000000e+00,2.000000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,8.000000e+01,3.131600e+04,2.000000e+00,2.000000e+00,6.200000e+01,1.230000e+02,3.700000e+01,2.000000e+00,3.400000e+01,0.000000e+00,...,1.000000e+00,2.400000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,4.430000e+02,3.204828e+06,5.000000e+00,4.000000e+00,1.870000e+02,4.820000e+02,8.100000e+01,3.600000e+01,5.000000e+01,2.616295e+01,...,2.000000e+00,3.200000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
max,6.553500e+04,1.200000e+08,2.197590e+05,2.919220e+05,1.290000e+07,6.554530e+08,2.482000e+04,2.325000e+03,5.940857e+03,7.125597e+03,...,2.135570e+05,1.380000e+02,1.100000e+08,7.420000e+07,1.100000e+08,1.100000e+08,1.200000e+08,7.690000e+07,1.200000e+08,1.200000e+08


In [10]:
df.columns.tolist()

[' Destination Port',
 ' Flow Duration',
 ' Total Fwd Packets',
 ' Total Backward Packets',
 'Total Length of Fwd Packets',
 ' Total Length of Bwd Packets',
 ' Fwd Packet Length Max',
 ' Fwd Packet Length Min',
 ' Fwd Packet Length Mean',
 ' Fwd Packet Length Std',
 'Bwd Packet Length Max',
 ' Bwd Packet Length Min',
 ' Bwd Packet Length Mean',
 ' Bwd Packet Length Std',
 'Flow Bytes/s',
 ' Flow Packets/s',
 ' Flow IAT Mean',
 ' Flow IAT Std',
 ' Flow IAT Max',
 ' Flow IAT Min',
 'Fwd IAT Total',
 ' Fwd IAT Mean',
 ' Fwd IAT Std',
 ' Fwd IAT Max',
 ' Fwd IAT Min',
 'Bwd IAT Total',
 ' Bwd IAT Mean',
 ' Bwd IAT Std',
 ' Bwd IAT Max',
 ' Bwd IAT Min',
 'Fwd PSH Flags',
 ' Bwd PSH Flags',
 ' Fwd URG Flags',
 ' Bwd URG Flags',
 ' Fwd Header Length',
 ' Bwd Header Length',
 'Fwd Packets/s',
 ' Bwd Packets/s',
 ' Min Packet Length',
 ' Max Packet Length',
 ' Packet Length Mean',
 ' Packet Length Std',
 ' Packet Length Variance',
 'FIN Flag Count',
 ' SYN Flag Count',
 ' RST Flag Count',
 ' P

In [11]:
# delete any space and turn features to lowercase
df.columns = df.columns.str.strip().str.lower()

In [12]:
df.columns.tolist()

['destination port',
 'flow duration',
 'total fwd packets',
 'total backward packets',
 'total length of fwd packets',
 'total length of bwd packets',
 'fwd packet length max',
 'fwd packet length min',
 'fwd packet length mean',
 'fwd packet length std',
 'bwd packet length max',
 'bwd packet length min',
 'bwd packet length mean',
 'bwd packet length std',
 'flow bytes/s',
 'flow packets/s',
 'flow iat mean',
 'flow iat std',
 'flow iat max',
 'flow iat min',
 'fwd iat total',
 'fwd iat mean',
 'fwd iat std',
 'fwd iat max',
 'fwd iat min',
 'bwd iat total',
 'bwd iat mean',
 'bwd iat std',
 'bwd iat max',
 'bwd iat min',
 'fwd psh flags',
 'bwd psh flags',
 'fwd urg flags',
 'bwd urg flags',
 'fwd header length',
 'bwd header length',
 'fwd packets/s',
 'bwd packets/s',
 'min packet length',
 'max packet length',
 'packet length mean',
 'packet length std',
 'packet length variance',
 'fin flag count',
 'syn flag count',
 'rst flag count',
 'psh flag count',
 'ack flag count',
 'ur

In [13]:
df["label"].value_counts()

label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64

In [14]:
class_percentage = df["label"].value_counts(normalize = True)* 100
print(class_percentage)

label
BENIGN                        80.300366
DoS Hulk                       8.162981
PortScan                       5.614427
DDoS                           4.522735
DoS GoldenEye                  0.363615
FTP-Patator                    0.280421
SSH-Patator                    0.208320
DoS slowloris                  0.204752
DoS Slowhttptest               0.194260
Bot                            0.069452
Web Attack � Brute Force       0.053237
Web Attack � XSS               0.023033
Infiltration                   0.001272
Web Attack � Sql Injection     0.000742
Heartbleed                     0.000389
Name: proportion, dtype: float64


In [15]:
df.duplicated().sum()

np.int64(308381)

In [16]:
duplicate_rows = df[df.duplicated()]

In [17]:
duplicate_rows

,destination port,flow duration,total fwd packets,total backward packets,total length of fwd packets,total length of bwd packets,fwd packet length max,fwd packet length min,fwd packet length mean,fwd packet length std,...,min_seg_size_forward,active mean,active std,active max,active min,idle mean,idle std,idle max,idle min,label
2109,80,77,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2257,443,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2749,443,49,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2862,443,4,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2877,443,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2830701,53,179,2,2,46,46,23,23,23.0,0.0,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2830725,53,161,2,2,82,114,41,41,41.0,0.0,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2830726,53,212,2,2,84,162,42,42,42.0,0.0,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2830731,443,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [18]:
for label_name in df["label"].unique():
    class_subset = df[df["label"] == label_name]

    dups_count = class_subset.duplicated().sum()
    total_count = len(class_subset)

    print(
        f"Class: [{label_name}] --> Total: {total_count} rows | Duplicates: {dups_count} rows"
    )

Class: [BENIGN] --> Total: 2273097 rows | Duplicates: 176613 rows
Class: [DDoS] --> Total: 128027 rows | Duplicates: 11 rows
Class: [PortScan] --> Total: 158930 rows | Duplicates: 68111 rows
Class: [Bot] --> Total: 1966 rows | Duplicates: 13 rows
Class: [Infiltration] --> Total: 36 rows | Duplicates: 0 rows
Class: [Web Attack � Brute Force] --> Total: 1507 rows | Duplicates: 37 rows
Class: [Web Attack � XSS] --> Total: 652 rows | Duplicates: 0 rows
Class: [Web Attack � Sql Injection] --> Total: 21 rows | Duplicates: 0 rows
Class: [FTP-Patator] --> Total: 7938 rows | Duplicates: 2005 rows
Class: [SSH-Patator] --> Total: 5897 rows | Duplicates: 2678 rows
Class: [DoS slowloris] --> Total: 5796 rows | Duplicates: 411 rows
Class: [DoS Slowhttptest] --> Total: 5499 rows | Duplicates: 271 rows
Class: [DoS Hulk] --> Total: 231073 rows | Duplicates: 58224 rows
Class: [DoS GoldenEye] --> Total: 10293 rows | Duplicates: 7 rows
Class: [Heartbleed] --> Total: 11 rows | Duplicates: 0 rows


In [19]:
df_clean = df.copy()

In [20]:
benign_class_name = "BENIGN"  

df_benign = df_clean[df_clean["label"] == benign_class_name]
df_attacks = df_clean[df_clean["label"] != benign_class_name]

print(f"Benign: {len(df_benign)}")
print(f"Attack: {len(df_attacks)}")

Benign: 2273097
Attack: 557646


In [21]:
df_clean.duplicated().sum()

np.int64(308381)

In [26]:
df_clean = df_clean.drop_duplicates()

In [27]:
df_clean.duplicated().sum()

np.int64(0)

In [29]:
# Function to Inspect Nulls and Inf Values
def inspect_inf_and_nan(df, label_col="label"):
    """Fast vectorized inspection for Inf and NaN values in dataframe features."""
    df_temp = df.copy()
    report = []
    total_rows = len(df_temp)

    # Select feature columns (exclude label column)
    feature_cols = [col for col in df_temp.columns if col != label_col]

    for col in feature_cols:
        # Fast vectorized numeric conversion
        num_series = pd.to_numeric(df_temp[col], errors="coerce")
        n_inf = np.isinf(num_series).sum()
        n_nan = num_series.isna().sum()

        if n_inf > 0 or n_nan > 0:
            report.append(
                {
                    "column": col,
                    "inf_count": n_inf,
                    "inf_%": round((n_inf / total_rows) * 100, 4),
                    "nan_count": n_nan,
                    "nan_%": round((n_nan / total_rows) * 100, 4),
                    "total_corrupted": n_inf + n_nan,
                }
            )

    report_df = pd.DataFrame(report)
    print("=" * 65)
    print("Report For Nulls And Infinity Values")
    print("=" * 65)

    if not report_df.empty:
        report_df = report_df.sort_values(by="total_corrupted", ascending=False)
        print(report_df.to_string(index=False))
    else:
        print("There are no NaN or Inf values found.")

    return report_df

In [30]:
report_df = inspect_inf_and_nan(df_clean, label_col="label")

Report For Nulls And Infinity Values
        column  inf_count  inf_%  nan_count  nan_%  total_corrupted
  flow bytes/s          0    0.0       1564  0.062             1564
flow packets/s          0    0.0       1564  0.062             1564


In [31]:
# ب. معالجة وتوحيد تحويل الأعمدة العدديدية وتحديد الـ Infs كـ NaNs شمولية
numeric_cols = [col for col in df_clean.columns if col != "label"]
for col in numeric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

# ج. تحديد قناع شامل لكل الصفوف التالفة (NaN أو Inf) عبر جميع الميزات
corrupted_mask = df_clean[numeric_cols].isna().any(axis=1) | np.isinf(
    df_clean[numeric_cols]
).any(axis=1)

In [32]:
# numbers of Nulls in each class in columne = label
print(df_clean[corrupted_mask]["label"].value_counts())

label
BENIGN         1427
PortScan        125
Bot               5
DoS Hulk          3
DDoS              2
FTP-Patator       2
Name: count, dtype: int64


In [33]:
df_clean['label'].value_counts()

label
BENIGN                        2096484
DoS Hulk                       172849
DDoS                           128016
PortScan                        90819
DoS GoldenEye                   10286
FTP-Patator                      5933
DoS slowloris                    5385
DoS Slowhttptest                 5228
SSH-Patator                      3219
Bot                              1953
Web Attack � Brute Force         1470
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64

In [34]:
print("Percentage Loss Per Class Before Dropping:")

total_per_class = df_clean["label"].value_counts()
corrupted_per_class = df_clean[corrupted_mask]["label"].value_counts()
percentage_loss = (corrupted_per_class / total_per_class) * 100
print(percentage_loss.dropna().round(4).astype(str) + " %")

Percentage Loss Per Class Before Dropping:
label
BENIGN         0.0681 %
Bot             0.256 %
DDoS           0.0016 %
DoS Hulk       0.0017 %
FTP-Patator    0.0337 %
PortScan       0.1376 %
Name: count, dtype: str


In [26]:
# Now we can remove nulls and we are reassured
# need to handle those tow columnes in Online Preprocessing Pipeline

In [35]:
df_clean.replace([np.inf, -np.inf], np.nan, inplace=True)
print(f"number of raws before drop: {len(df_clean):,}")

df_clean.dropna(inplace=True)
df_clean.reset_index(drop=True, inplace=True)

print(f"Number of rows after dropping nulls/infs: {len(df_clean):,}")

number of raws before drop: 2,522,362
Number of rows after dropping nulls/infs: 2,520,798


In [37]:
print(f"number of NaNs is {2522362 - 2520798}")

number of NaNs is 1564


# splitting

In [38]:
from sklearn.model_selection import train_test_split
y = df_clean['label'].apply(lambda x: 0 if str(x).strip().upper() == 'BENIGN' else 1)

In [39]:
if df_clean["fwd header length"].equals(df_clean["fwd header length.1"]): 
    print("YES") 
else: 
    print("NO")

YES


In [40]:
cols_to_drop = ['label', 'destination port', 'fwd header length.1']
X = df_clean.drop(columns=[col for col in cols_to_drop if col in df_clean.columns])

In [41]:
# (Stratified Train/Val/Test Split) 
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

In [42]:
# Train (70%) and Validation (15%)
# 15/85 = .17647
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.17647, random_state=42, stratify=y_train_val
)

In [43]:
print(f" {X_train.shape}")
print(f" {X_val.shape}")
print(f" {X_test.shape}")

 (1764559, 76)
 (378119, 76)
 (378120, 76)


In [44]:
X_train_prep = X_train.copy()
X_val_prep = X_val.copy()
X_test_prep = X_test.copy()

# Feature Selection

In [45]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from boruta import BorutaPy

# Pearson Correlation
def remove_pearson_redundant_features(X_data, threshold=0.90):
    corr_matrix = X_data.corr(method="pearson").abs()
    # take the upper triangle
    upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

    # Identify columns whose correlation exceeds the threshold
    to_drop = [
        column for column in upper_tri.columns 
        if any(upper_tri[column] > threshold)
    ]
    df_reduced = X_data.drop(columns=to_drop)

    print(
        f"were excluded {len(to_drop)} / reamining : {df_reduced.shape[1]}"
    )
    return df_reduced, to_drop

In [46]:
# MI-Boruta Hybrid Selection 
def run_mi_boruta_feature_selection(
    X_train, y_train, mi_threshold=0.01, sample_size=150000, max_iter=30
):
    """Executes MI-Boruta Hybrid Feature Selection combining Mutual Information 
    filtering with Boruta shadow feature testing.
    """    
    # Perform fast stratified subsampling if dataset exceeds sample_size
    if len(X_train) > sample_size:
        print(f" Subsampling {sample_size:,} records from training data for fast evaluation...")
        X_sample, _, y_sample, _ = train_test_split(
            X_train, y_train, train_size=sample_size, stratify=y_train, random_state=42
        )
    else:
        X_sample, y_sample = X_train, y_train

    #  Filter features using Mutual Information
    print(f"\n Calculating Mutual Information (filtering scores <= {mi_threshold})...")
    mi_scores = mutual_info_classif(
        X_sample, y_sample, discrete_features="auto", random_state=42
    )
    
    mi_series = pd.Series(mi_scores, index=X_sample.columns).sort_values(ascending=False)
    mi_passed_cols = mi_series[mi_series > mi_threshold].index.tolist()
    print(f"  Features passing MI pre-filter: {len(mi_passed_cols)} / {X_train.shape[1]}")

    # Fallback safety: If MI filters out too many features, retain top 25 features
    if len(mi_passed_cols) < 5:
        print(" Warning: Too few features passed MI threshold. Falling back to top 25 MI features.")
        mi_passed_cols = mi_series.head(25).index.tolist()
        
    X_mi_subset = X_sample[mi_passed_cols].copy()

    # Boruta Wrapper selection with Shadow Features
    print("\n Running Boruta Wrapper with Random Forest & Shadow Features...")
    rf = RandomForestClassifier(n_estimators=100, max_depth=7, n_jobs=-1, random_state=42)
    
    boruta_selector = BorutaPy(
        estimator=rf,
        n_estimators="auto",
        max_iter=max_iter,
        random_state=42,
        verbose=1,
    )
    
    boruta_selector.fit(X_mi_subset.values, y_sample.values)

    # Extract confirmed features
    confirmed_mask = boruta_selector.support_
    selected_cols = np.array(mi_passed_cols)[confirmed_mask].tolist()

    # Fallback safety: Include tentative (support_weak_) features if confirmed set is small
    if len(selected_cols) < 5:
        tentative_mask = boruta_selector.support_weak_
        tentative_cols = np.array(mi_passed_cols)[tentative_mask].tolist()
        selected_cols = list(set(selected_cols + tentative_cols))
        print(f" Included tentative Boruta features. Total selected: {len(selected_cols)}")

    # Ultimate fallback safety: If Boruta selects 0 features, use top 15 MI features
    if len(selected_cols) == 0:
        print(" Warning: Boruta confirmed 0 features. Falling back to top 15 MI features.")
        selected_cols = mi_series.head(15).index.tolist()

    print(f"\n MI-Boruta Selection Complete! Selected {len(selected_cols)} features:")
    for i, col in enumerate(selected_cols, 1):
        print(f"  {i}. {col}")
        
    return selected_cols

In [47]:
# Apply Pearson Redundancy Filter on Training Set Only
X_train_pearson, dropped_pearson = remove_pearson_redundant_features(
    X_train_prep, threshold=0.90
)

# Apply MI-Boruta Hybrid Feature Selection
selected_cols = run_mi_boruta_feature_selection(
    X_train_pearson, y_train, mi_threshold=0.01, sample_size=150000, max_iter=30
)

were excluded 30 / reamining : 46
 Subsampling 150,000 records from training data for fast evaluation...

 Calculating Mutual Information (filtering scores <= 0.01)...
  Features passing MI pre-filter: 35 / 46

 Running Boruta Wrapper with Random Forest & Shadow Features...
Iteration: 1 / 30
Iteration: 2 / 30
Iteration: 3 / 30
Iteration: 4 / 30
Iteration: 5 / 30
Iteration: 6 / 30
Iteration: 7 / 30
Iteration: 8 / 30


BorutaPy finished running.

Iteration: 	9 / 30
Confirmed: 	35
Tentative: 	0
Rejected: 	0

 MI-Boruta Selection Complete! Selected 35 features:
  1. total length of fwd packets
  2. bwd packet length max
  3. init_win_bytes_backward
  4. init_win_bytes_forward
  5. fwd packet length max
  6. fwd packet length mean
  7. fwd header length
  8. flow duration
  9. flow bytes/s
  10. bwd packets/s
  11. bwd header length
  12. flow packets/s
  13. flow iat mean
  14. bwd iat max
  15. flow iat std
  16. bwd iat total
  17. active mean
  18. active max
  19. bwd iat mean
  20. to

# scaling

In [117]:
from sklearn.preprocessing import RobustScaler
df_clean['fin flag count'].value_counts()

fin flag count
0    2439892
1      80906
Name: count, dtype: int64

In [118]:
# Robust Pre-Scaling, Dynamic Log1p Transform & Robust Scaling Pipeline
# Subset selected features
X_train_sub = X_train_prep[selected_cols].copy()
X_val_sub = X_val_prep[selected_cols].copy()
X_test_sub = X_test_prep[selected_cols].copy()

In [119]:
#  Coerce data types and replace extreme out-of-bounds float values with NaN
max_float_limit = np.finfo(np.float64).max / 10

for df_sub in [X_train_sub, X_val_sub, X_test_sub]:
    for col in df_sub.columns:
        df_sub[col] = pd.to_numeric(df_sub[col], errors="coerce")
        df_sub[col] = df_sub[col].mask(np.abs(df_sub[col]) >= max_float_limit, np.nan)
print("max num is: ", np.nanmax(X_train_sub.values))

max num is:  2071000000.0


In [120]:
# Impute missing/corrupted values using X_train medians (Prevents Data Leakage)
train_medians = X_train_sub.median().fillna(0)

X_train_sub.fillna(train_medians, inplace=True)
X_val_sub.fillna(train_medians, inplace=True)
X_test_sub.fillna(train_medians, inplace=True)

,total length of fwd packets,bwd packet length max,init_win_bytes_backward,init_win_bytes_forward,fwd packet length max,fwd packet length mean,fwd header length,flow duration,flow bytes/s,bwd packets/s,...,flow iat min,act_data_pkt_fwd,min_seg_size_forward,down/up ratio,fin flag count,idle std,urg flag count,psh flag count,active std,ack flag count
136684,76,326,-1,-1,38,38.000000,64,299,2.434783e+06,6688.963211,...,3,1,32,1,0,0.0,0,0,0.0,0
239336,12,0,29200,8192,6,4.000000,72,5788954,2.072913e+00,0.172743,...,93,2,20,0,0,0.0,0,1,0.0,0
610341,254,0,-1,30,217,84.666667,60,866,2.933025e+05,0.000000,...,4,2,20,0,0,0.0,0,0,0.0,1
1628679,887,2896,122,65535,405,32.851852,876,154725,3.467119e+05,219.744708,...,1,6,32,1,0,0.0,0,1,0.0,0
94758,1375,6,0,229,1375,1375.000000,20,9496466,1.479498e+02,0.631814,...,1,0,20,6,0,0.0,1,0,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141801,26,7215,229,8192,20,8.666667,72,15225,7.640722e+05,328.407225,...,3,2,20,1,0,0.0,0,1,0.0,0
1976239,1086,482,62,8192,1080,362.000000,72,241745,6.486173e+03,8.273180,...,119,2,20,0,0,0.0,0,1,0.0,0
1366464,37,0,-1,121,37,18.500000,64,3,1.230000e+07,0.000000,...,3,0,32,0,0,0.0,0,0,0.0,1
151255,26,8760,229,8192,20,8.666667,72,996314,1.167002e+04,4.014799,...,82,2,20,1,0,0.0,0,1,0.0,0


In [121]:
# identify highly skewed features on X_train (Skewness > 2.0)
skewness_scores = X_train_sub.skew().abs()
high_skew_cols = skewness_scores[skewness_scores > 2].index.tolist()
print(f" Applied Log1p transform to {len(high_skew_cols)} highly skewed features.")

 Applied Log1p transform to 32 highly skewed features.


In [122]:
all_skewness = X_train_sub.skew().abs()
high_skew_cols = all_skewness[all_skewness > 2].index.tolist()
escaped_cols = all_skewness[all_skewness <= 2].index.tolist()

print(f" (Skew > 2): {len(high_skew_cols)}")
print(f" (Skew <= 2): {len(escaped_cols)}\n")
print("(Escaped Columns):")

report_data = []
for col in escaped_cols:
    max_val = X_train_sub[col].max()
    min_val = X_train_sub[col].min()
    skew_val = all_skewness[col]

    report_data.append(
        {
            "Column Name": col,
            "Skewness": round(skew_val, 4),
            "Max Value": max_val,
            "Min Value": min_val,
        }
    )

escaped_df = pd.DataFrame(report_data).sort_values(by="Max Value", ascending=False)
print(escaped_df.to_string(index=False, float_format="{:,.2f}".format))

 (Skew > 2): 32
 (Skew <= 2): 3

(Escaped Columns):
   Column Name  Skewness  Max Value  Min Value
 flow duration      1.96  119999997        -12
psh flag count      0.89          1          0
ack flag count      0.81          1          0


In [123]:
forced_log_cols = [
    col
    for col in X_train_sub.columns
    if any(
        k in col.lower()
        for k in [
            "bytes",
            "packets",
            "duration",
            "iat",
            "length",
            "size",
            "active",
            "idle",
        ]
    )
]

final_log_cols = list(set(high_skew_cols + forced_log_cols))

In [125]:
#  Apply Log1p (log(1+x)) transformation to stabilize variance
for col in final_log_cols:
    X_train_sub[col] = np.log1p(X_train_sub[col].clip(lower=0))
    X_val_sub[col] = np.log1p(X_val_sub[col].clip(lower=0))
    X_test_sub[col] = np.log1p(X_test_sub[col].clip(lower=0))

In [126]:
print("max num is: ", np.nanmax(X_train_sub.values))

max num is:  21.451297419842717


In [127]:
# Fit RobustScaler strictly on X_train and transform all partitions
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_sub)
X_val_scaled = scaler.transform(X_val_sub)
X_test_scaled = scaler.transform(X_test_sub)

In [128]:
print(" Preprocessing and Robust Scaling completed successfully!")
print(f" - Final Scaled Train Shape: {X_train_scaled.shape}")
print(f" - Any remaining NaNs in Train: {np.isnan(X_train_scaled).any()}")
print(f" - Any remaining Infs in Train: {np.isinf(X_train_scaled).any()}")
print("max num is: ", np.nanmax(X_train_scaled))

 Preprocessing and Robust Scaling completed successfully!
 - Final Scaled Train Shape: (1764559, 35)
 - Any remaining NaNs in Train: False
 - Any remaining Infs in Train: False
max num is:  18.5159909328476


# training

In [62]:
# from src.evaluation import convert_iforest_scores, iforest_predictions_to_binary, print_metrics_report, plot_roc_and_pr, log_experiment

# scores = convert_iforest_scores(iso_model, X_test)          # higher = more anomalous
# preds  = iforest_predictions_to_binary(iso_model, X_test)   # 1 = anomaly, 0 = normal

# metrics = print_metrics_report(y_test, preds, scores, model_name="Isolation Forest")
# plot_roc_and_pr(y_test, scores, model_name="Isolation Forest")

# log_experiment("../experiments/results_log.csv", "IsolationForest",
#                 params={"n_estimators": 200, "contamination": 0.03},
#                 metrics=metrics)

In [131]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    auc,
    classification_report,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)

# STEP 4: Isolation Forest Training & Validation Threshold Optimization

# 1. Train strictly on BENIGN baseline traffic (y_train == 0)
print(" Subsetting BENIGN baseline traffic for training...")
benign_mask = np.array(y_train) == 0
X_train_benign = X_train_scaled[benign_mask]

print(
    f" - Total Train Samples: {X_train_scaled.shape[0]:,} | BENIGN Baseline Samples: {X_train_benign.shape[0]:,}"
)

# 2. Initialize Isolation Forest with tuned hyperparameters
iso_model = IsolationForest(
    n_estimators=400,  # 400 trees for stable score distribution
    max_samples=512,  # 512 samples per tree to optimize split resolution
    max_features=0.8,  # Subsample features for diversity
    contamination="auto",
    random_state=42,
    n_jobs=-1,
)

iso_model.fit(X_train_benign)

# 3. Calculate continuous anomaly scores on Validation Set (-decision_function)
val_scores = -iso_model.decision_function(X_val_scaled)

# 4. Compute Precision-Recall Curve & PR-AUC on Validation Set
precisions, recalls, thresholds = precision_recall_curve(y_val, val_scores)
val_pr_auc = auc(recalls, precisions)
val_roc_auc = roc_auc_score(y_val, val_scores)

print("\n Validation Global Curve Metrics:")
print(f" - Validation PR-AUC (AUC-PR): {val_pr_auc:.4f}")
print(f" - Validation ROC-AUC:         {val_roc_auc:.4f}")

# 5. Search for Optimal Threshold on PR-Curve (Maximizing F1-Score or Target SLA)
print("\n Searching for Optimal Decision Threshold on PR-Curve...")
best_thresh = None
best_f1 = 0.0
target_thresh = None
target_best_f1 = 0.0

for p, r, thresh in zip(precisions[:-1], recalls[:-1], thresholds):
    f1 = (2 * p * r) / (p + r) if (p + r) > 0 else 0.0

    # Track global highest F1 threshold
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thresh

    # Track specific SLA target threshold (Precision >= 85% & Recall >= 80%)
    if p >= 0.85 and r >= 0.80:
        if target_thresh is None or f1 > target_best_f1:
            target_best_f1 = f1
            target_thresh = thresh

# Use SLA threshold if met; otherwise fallback to maximum F1 threshold
selected_thresh = target_thresh if target_thresh is not None else best_thresh
val_preds = (val_scores >= selected_thresh).astype(int)

print(f"\n Optimal Frozen Threshold Selected: {selected_thresh:.4f}")
print(f" - Validation F1-Score:  {f1_score(y_val, val_preds, zero_division=0):.4f}")
print(
    f" - Validation Precision: {precision_score(y_val, val_preds, zero_division=0):.4f}"
)
print(f" - Validation Recall:    {recall_score(y_val, val_preds, zero_division=0):.4f}")

print("\n Validation Classification Report:")
print(
    classification_report(
        y_val, val_preds, target_names=["BENIGN (0)", "Attack (1)"], digits=4
    )
)

# STEP 5: Final Independent Evaluation on Unseen Test Set

print("\n🧪 Evaluating on Unseen Test Set using Frozen Validation Threshold...")
test_scores = -iso_model.decision_function(X_test_scaled)
test_preds = (test_scores >= selected_thresh).astype(int)

# Test Metrics
test_precisions, test_recalls, _ = precision_recall_curve(y_test, test_scores)
test_pr_auc = auc(test_recalls, test_precisions)
test_roc_auc = roc_auc_score(y_test, test_scores)

print("\n Final Test Set Benchmark Metrics:")
print(f" - Test PR-AUC (AUC-PR): {test_pr_auc:.4f}")
print(f" - Test ROC-AUC:         {test_roc_auc:.4f}")
print(f" - Test F1-Score:        {f1_score(y_test, test_preds, zero_division=0):.4f}")
print(
    f" - Test Precision:       {precision_score(y_test, test_preds, zero_division=0):.4f}"
)
print(
    f" - Test Recall:          {recall_score(y_test, test_preds, zero_division=0):.4f}"
)

print("\n    Test Set Classification Report:")
print(
    classification_report(
        y_test, test_preds, target_names=["BENIGN (0)", "Attack (1)"], digits=4
    )
)

 Subsetting BENIGN baseline traffic for training...
 - Total Train Samples: 1,764,559 | BENIGN Baseline Samples: 1,466,540

 Validation Global Curve Metrics:
 - Validation PR-AUC (AUC-PR): 0.4398
 - Validation ROC-AUC:         0.8776

 Searching for Optimal Decision Threshold on PR-Curve...

 Optimal Frozen Threshold Selected: 0.0197
 - Validation F1-Score:  0.6223
 - Validation Precision: 0.5492
 - Validation Recall:    0.7178

 Validation Classification Report:
              precision    recall  f1-score   support

  BENIGN (0)     0.9388    0.8803    0.9086    314258
  Attack (1)     0.5492    0.7178    0.6223     63861

    accuracy                         0.8528    378119
   macro avg     0.7440    0.7990    0.7655    378119
weighted avg     0.8730    0.8528    0.8603    378119


🧪 Evaluating on Unseen Test Set using Frozen Validation Threshold...

 Final Test Set Benchmark Metrics:
 - Test PR-AUC (AUC-PR): 0.4421
 - Test ROC-AUC:         0.8786
 - Test F1-Score:        0.6238
 - 

In [ ]:
import joblib
import numpy as np

pipeline_assets = {
    "selected_cols": selected_cols,
    "scaler": scaler,
    "iso_model": iso_model,
    "selected_thresh": selected_thresh,
}
joblib.dump(pipeline_assets, "nids_iso_forest_pipeline.pkl")
print(" model and paramters have been saved into nids_iso_forest_pipeline.pkl!")

np.savez_compressed(
    "scaled_datasets.npz",
    X_train=X_train_scaled,
    X_val=X_val_scaled,
    X_test=X_test_scaled,
    y_train=y_train,
    y_val=y_val,
    y_test=y_test,
)
print("✅Measured data sets were successfully saved in scaled_datasets.npz!")

✅ تم حفظ الموديل والبرامترات بنجاح في nids_iso_forest_pipeline.pkl!
✅Measured data sets were successfully saved in scaled_datasets.npz!
